# FaceGuard: One-Click Colab Experiment Notebook

This notebook runs the full FaceGuard experimental pipeline:

1. Install dependencies.
2. Load a reliable face dataset (LFW via sklearn; fallback to skimage LFW subset).
3. Run FaceGuard with ablation variants:
   - Random-Noise-Sanity
   - Ablation-FGSM-1Step
   - Ablation-PGD-NoEOT
   - FaceGuard-EOT
4. Evaluate identity cosine similarity, PSNR, SSIM, JPEG robustness, Gaussian blur robustness, and resize/downscale robustness.
5. Optionally evaluate with a real face-swap model using InsightFace InSwapper if `inswapper_128.onnx` is available at `/content/inswapper_128.onnx`.
6. Export CSV files, figures, and visual examples into `/content/faceguard_outputs`.

To run: open in Google Colab and click **Runtime → Run all**.

In [5]:
# ============================================================
# FaceGuard One-Click Colab
# Dependency installation
# ============================================================

!uv pip -q install facenet-pytorch insightface scikit-learn scikit-image pandas tqdm matplotlib opencv-python pillow 

In [ ]:
!pip install --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 16.0 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 91.9 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 56.4 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 105.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 51.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 140.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 20.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 27.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 71.3 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 93.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12

In [ ]:
!apt update && apt install -y --no-install-recommends libxcb1 libx11-6 libxext6 libxrender1 libsm6 libxrandr2 libgl1 libglib2.0-0 libgomp1

Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Hit:2 http://security.ubuntu.com/ubuntu noble-security InRelease
Hit:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease
Hit:4 http://archive.ubuntu.com/ubuntu noble-backports InRelease
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
10 packages can be upgraded. Run 'apt list --upgradable' to see them.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Note, selecting 'libglib2.0-0t64' instead of 'libglib2.0-0'
libxcb1 is already the newest version (1.15-1ubuntu2).
libx11-6 is already the newest version (2:1.8.7-1build1).
libxext6 is already the newest version (2:1.3.4-1build2).
libxrender1 is already the newest version (1:0.9.10-1.1build1).
libsm6 is already the newest version (2:1.2.3-1build3).
libxrandr2 is already the newest version (2:1.5.2-2build1).
libgl1 is already the newest version (1.7.0-1build1).
libgomp1 is already the

In [ ]:
# ============================================================
# FaceGuard: Invisible Adversarial Defenses for Protecting Privacy
# Against Deepfake Identity Reconstruction
#
# One-click experimental pipeline:
# - dataset loading
# - FaceGuard perturbation
# - ablation study
# - robustness evaluation
# - optional real deepfake evaluation using InsightFace InSwapper
# - CSV/figure/visual export
# ============================================================

import os
import cv2
import math
import random
import shutil
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from PIL import Image

import torch
import torch.nn.functional as F

from facenet_pytorch import InceptionResnetV1
from skimage.metrics import structural_similarity as skimage_ssim

# ============================================================
# 0. Global configuration
# ============================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("[*] Device:", device)

OUTPUT_DIR = "/content/faceguard_outputs"
VIS_DIR = os.path.join(OUTPUT_DIR, "visual_examples")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)

# Dataset settings:
# DATASET_MODE = "lfw" uses sklearn LFW if available and falls back to skimage LFW subset.
# DATASET_MODE = "folder" reads images from CUSTOM_FOLDER.
DATASET_MODE = "lfw"
CUSTOM_FOLDER = "/content/faceguard_dataset"

# For quick test, use 30-80. For paper-scale, increase to 200-1000+.
MAX_PAIRS = 80
IMAGE_SIZE = 224

# Experimental settings
SEEDS = [0, 1, 2, 3, 4]
EPSILON_LIST = [4/255, 8/255, 12/255, 16/255]
ALPHA_RATIOS = [0.10, 0.125, 0.15]
STEPS = 35

# FaceGuard robust objective
USE_EOT_FOR_FACEGUARD = True
EOT_WEIGHT = 0.50

# Real deepfake evaluation settings
# Upload inswapper_128.onnx to /content/inswapper_128.onnx to enable.
RUN_REAL_DEEPFAKE = True
SWAPPER_MODEL_PATH = "/content/inswapper_128.onnx"
MAX_DEEPFAKE_EVAL_PAIRS = 30
SAVE_VISUAL_EXAMPLES = 10

print("[*] Output directory:", OUTPUT_DIR)
print("[*] Real deepfake model path:", SWAPPER_MODEL_PATH)


# ============================================================
# 1. Utility functions
# ============================================================

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def ensure_uint8_rgb(img):
    if isinstance(img, Image.Image):
        img = np.array(img.convert("RGB"))
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    if img.max() <= 1.0:
        img = (img * 255).astype(np.uint8)
    return img.astype(np.uint8)


def resize_rgb(img, size=224):
    img = ensure_uint8_rgb(img)
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)


def rgb_to_tensor(img_rgb):
    img_rgb = ensure_uint8_rgb(img_rgb)
    x = torch.from_numpy(img_rgb).permute(2, 0, 1).float() / 255.0
    return x.unsqueeze(0).to(device)


def tensor_to_rgb_uint8(x):
    x = x.detach().clamp(0, 1).squeeze(0).permute(1, 2, 0).cpu().numpy()
    return (x * 255).round().astype(np.uint8)


def rgb_to_bgr(img_rgb):
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)


def bgr_to_rgb(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def psnr_tensor(x, y):
    mse = torch.mean((x - y) ** 2).item()
    if mse <= 1e-12:
        return 100.0
    return 20 * math.log10(1.0 / math.sqrt(mse))


def psnr_np_rgb(a, b):
    a = ensure_uint8_rgb(a).astype(np.float32) / 255.0
    b = ensure_uint8_rgb(b).astype(np.float32) / 255.0
    mse = np.mean((a - b) ** 2)
    if mse <= 1e-12:
        return 100.0
    return 20 * math.log10(1.0 / math.sqrt(mse))


def ssim_np_rgb(a, b):
    a = ensure_uint8_rgb(a)
    b = ensure_uint8_rgb(b)
    try:
        return skimage_ssim(a, b, channel_axis=2, data_range=255)
    except TypeError:
        return skimage_ssim(a, b, multichannel=True, data_range=255)


def make_face_mask(h=224, w=224, mode="ellipse"):
    M = torch.zeros((1, 1, h, w), device=device)

    if mode == "rect":
        y1, y2 = int(0.16*h), int(0.84*h)
        x1, x2 = int(0.20*w), int(0.80*w)
        M[:, :, y1:y2, x1:x2] = 1.0

    elif mode == "ellipse":
        yy, xx = torch.meshgrid(
            torch.arange(h, device=device),
            torch.arange(w, device=device),
            indexing="ij"
        )
        cy, cx = int(0.52*h), int(0.50*w)
        ry, rx = int(0.36*h), int(0.31*w)
        mask = (((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2) <= 1.0
        M[:, :, mask] = 1.0

    else:
        M[:] = 1.0

    return M


def simulate_jpeg_tensor(x, quality=75):
    img = tensor_to_rgb_uint8(x)
    bgr = rgb_to_bgr(img)
    ok, enc = cv2.imencode(".jpg", bgr, [int(cv2.IMWRITE_JPEG_QUALITY), quality])
    if not ok:
        return x.detach()
    dec = cv2.imdecode(enc, cv2.IMREAD_COLOR)
    rgb = bgr_to_rgb(dec)
    return rgb_to_tensor(rgb)


def simulate_blur_tensor(x, ksize=5):
    img = tensor_to_rgb_uint8(x)
    img_blur = cv2.GaussianBlur(img, (ksize, ksize), 0)
    return rgb_to_tensor(img_blur)


def simulate_resize_tensor(x, scale=0.80):
    img = tensor_to_rgb_uint8(x)
    h, w = img.shape[:2]
    small = cv2.resize(img, (max(1, int(w*scale)), max(1, int(h*scale))), interpolation=cv2.INTER_AREA)
    back = cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)
    return rgb_to_tensor(back)


def differentiable_resize(x, scale=0.80):
    h, w = x.shape[-2:]
    small = F.interpolate(x, scale_factor=scale, mode="bilinear", align_corners=False)
    back = F.interpolate(small, size=(h, w), mode="bilinear", align_corners=False)
    return back


def get_gaussian_kernel(k=5, sigma=1.0, channels=3):
    ax = torch.arange(k, device=device).float() - k // 2
    xx, yy = torch.meshgrid(ax, ax, indexing="ij")
    kernel = torch.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    kernel = kernel / kernel.sum()
    kernel = kernel.view(1, 1, k, k)
    kernel = kernel.repeat(channels, 1, 1, 1)
    return kernel


def differentiable_blur(x, k=5, sigma=1.0):
    kernel = get_gaussian_kernel(k=k, sigma=sigma, channels=x.shape[1])
    pad = k // 2
    return F.conv2d(x, kernel, padding=pad, groups=x.shape[1])


def save_image_grid(images, titles, save_path, suptitle=None, figsize=(18, 4)):
    fig, axes = plt.subplots(1, len(images), figsize=figsize)
    if len(images) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(ensure_uint8_rgb(img))
        ax.set_title(title)
        ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=11)
    plt.savefig(save_path, dpi=250, bbox_inches="tight")
    plt.show()
    plt.close(fig)


# ============================================================
# 2. Dataset loading
# ============================================================

def load_lfw_pairs(max_pairs=80, image_size=224):
    # Tries sklearn LFW first, falls back to skimage LFW subset if download fails.
    print("[*] Loading LFW dataset...")

    try:
        from sklearn.datasets import fetch_lfw_people
        lfw = fetch_lfw_people(color=True, resize=1.0, min_faces_per_person=2)
        images = lfw.images
        labels = lfw.target

        rgb_images = []
        for img in images:
            img = ensure_uint8_rgb(img)
            img = resize_rgb(img, image_size)
            rgb_images.append(img)

        pairs = []
        n = len(rgb_images)
        indices = list(range(n))
        random.shuffle(indices)

        for i in indices:
            for j in indices:
                if labels[i] != labels[j]:
                    pairs.append({
                        "source_rgb": rgb_images[i],
                        "target_rgb": rgb_images[j],
                        "source_label": int(labels[i]),
                        "target_label": int(labels[j])
                    })
                    break
            if len(pairs) >= max_pairs:
                break

        print(f"[*] Loaded {len(pairs)} LFW source-target pairs from sklearn.")
        return pairs

    except Exception as e:
        print("[!] sklearn LFW failed. Falling back to skimage.data.lfw_subset().")
        print("    Reason:", repr(e))

        from skimage import data
        lfw_images = data.lfw_subset()
        rgb_images = []

        for img in lfw_images:
            img = ensure_uint8_rgb(img)
            img = resize_rgb(img, image_size)
            rgb_images.append(img)

        pairs = []
        usable = min(max_pairs, len(rgb_images)-1)
        for k in range(usable):
            pairs.append({
                "source_rgb": rgb_images[k],
                "target_rgb": rgb_images[(k+1) % len(rgb_images)],
                "source_label": k,
                "target_label": (k+1) % len(rgb_images)
            })

        print(f"[*] Loaded {len(pairs)} fallback LFW-subset pairs.")
        return pairs


def load_folder_pairs(folder, max_pairs=80, image_size=224):
    exts = [".jpg", ".jpeg", ".png", ".webp", ".bmp"]
    paths = [
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if os.path.splitext(f.lower())[1] in exts
    ]

    if len(paths) < 2:
        raise ValueError(f"Folder {folder} needs at least two face images.")

    random.shuffle(paths)
    imgs = []
    for p in paths:
        img = Image.open(p).convert("RGB")
        img = resize_rgb(np.array(img), image_size)
        imgs.append((p, img))

    pairs = []
    for k in range(min(max_pairs, len(imgs)-1)):
        src_path, src = imgs[k]
        tgt_path, tgt = imgs[(k+1) % len(imgs)]
        pairs.append({
            "source_rgb": src,
            "target_rgb": tgt,
            "source_label": os.path.basename(src_path),
            "target_label": os.path.basename(tgt_path)
        })

    print(f"[*] Loaded {len(pairs)} custom folder source-target pairs.")
    return pairs


set_seed(0)

if DATASET_MODE == "lfw":
    pairs = load_lfw_pairs(MAX_PAIRS, IMAGE_SIZE)
elif DATASET_MODE == "folder":
    pairs = load_folder_pairs(CUSTOM_FOLDER, MAX_PAIRS, IMAGE_SIZE)
else:
    raise ValueError("DATASET_MODE must be either 'lfw' or 'folder'.")

if len(pairs) == 0:
    raise RuntimeError("No valid source-target pairs were loaded.")

print("[*] Example source-target shapes:", pairs[0]["source_rgb"].shape, pairs[0]["target_rgb"].shape)


# ============================================================
# 3. Identity extractor
# ============================================================

print("[*] Loading identity extractor: InceptionResnetV1 pretrained on VGGFace2...")
identity_model = InceptionResnetV1(pretrained="vggface2").eval().to(device)
for p in identity_model.parameters():
    p.requires_grad = False
print("[*] Identity extractor loaded.")


def facenet_preprocess(x):
    # x: [B,3,H,W], range [0,1]
    # facenet-pytorch InceptionResnetV1 expects 160x160 and standardized input.
    x = F.interpolate(x, size=(160, 160), mode="bilinear", align_corners=False)
    x = (x - 0.5) / 0.5
    return x


@torch.no_grad()
def identity_embedding_no_grad(x):
    z = identity_model(facenet_preprocess(x))
    z = F.normalize(z, p=2, dim=1)
    return z


def identity_embedding_grad(x):
    z = identity_model(facenet_preprocess(x))
    z = F.normalize(z, p=2, dim=1)
    return z


def cosine_identity(x, y):
    zx = identity_embedding_no_grad(x)
    zy = identity_embedding_no_grad(y)
    return F.cosine_similarity(zx, zy, dim=1).item()


# ============================================================
# 4. FaceGuard and ablation methods
# ============================================================

def faceguard_loss(adv, z_orig, use_eot=True, eot_weight=0.50):
    z_adv = identity_embedding_grad(adv)
    loss_main = F.cosine_similarity(z_adv, z_orig, dim=1).mean()

    if not use_eot:
        return loss_main

    adv_resize = differentiable_resize(adv, scale=0.80)
    adv_blur = differentiable_blur(adv, k=5, sigma=1.0)

    z_resize = identity_embedding_grad(adv_resize)
    z_blur = identity_embedding_grad(adv_blur)

    loss_resize = F.cosine_similarity(z_resize, z_orig, dim=1).mean()
    loss_blur = F.cosine_similarity(z_blur, z_orig, dim=1).mean()

    return loss_main + eot_weight * 0.5 * (loss_resize + loss_blur)


def project_linf(adv, clean, eps):
    adv = clean + torch.clamp(adv - clean, min=-eps, max=eps)
    adv = torch.clamp(adv, 0.0, 1.0)
    return adv


def protect_faceguard(x, eps, alpha_ratio=0.125, steps=35, mask=None, use_eot=True):
    # Proposed method: FaceGuard-EOT.
    if mask is None:
        mask = torch.ones((1, 1, x.shape[-2], x.shape[-1]), device=device)

    alpha = eps * alpha_ratio

    with torch.no_grad():
        z_orig = identity_embedding_no_grad(x).detach()
        adv = x + torch.empty_like(x).uniform_(-eps, eps)
        adv = project_linf(adv, x, eps)

    for _ in range(steps):
        adv.requires_grad_(True)
        loss = faceguard_loss(adv, z_orig, use_eot=use_eot, eot_weight=EOT_WEIGHT)

        identity_model.zero_grad(set_to_none=True)
        if adv.grad is not None:
            adv.grad.zero_()
        loss.backward()

        with torch.no_grad():
            grad = adv.grad.sign()
            adv = adv - alpha * mask * grad
            adv = project_linf(adv, x, eps).detach()

    return adv


def ablation_random_noise(x, eps, mask=None):
    if mask is None:
        mask = torch.ones((1, 1, x.shape[-2], x.shape[-1]), device=device)
    noise = torch.empty_like(x).uniform_(-eps, eps) * mask
    return torch.clamp(x + noise, 0, 1)


def ablation_fgsm(x, eps, alpha_ratio=0.125, mask=None):
    # Ablation-FGSM-1Step: single-step identity disruption.
    if mask is None:
        mask = torch.ones((1, 1, x.shape[-2], x.shape[-1]), device=device)

    alpha = eps * alpha_ratio

    with torch.no_grad():
        z_orig = identity_embedding_no_grad(x).detach()

    adv = x.clone().detach().requires_grad_(True)
    z_adv = identity_embedding_grad(adv)
    loss = F.cosine_similarity(z_adv, z_orig, dim=1).mean()

    identity_model.zero_grad(set_to_none=True)
    loss.backward()

    with torch.no_grad():
        adv = adv - alpha * mask * adv.grad.sign()
        adv = project_linf(adv, x, eps)

    return adv.detach()


def ablation_pgd_no_eot(x, eps, alpha_ratio=0.125, steps=35, mask=None):
    # Ablation-PGD-NoEOT:
    # iterative identity-disruption optimization without transformation-aware objective.
    if mask is None:
        mask = torch.ones((1, 1, x.shape[-2], x.shape[-1]), device=device)

    alpha = eps * alpha_ratio

    with torch.no_grad():
        z_orig = identity_embedding_no_grad(x).detach()
        adv = x.clone().detach()

    for _ in range(steps):
        adv.requires_grad_(True)
        z_adv = identity_embedding_grad(adv)
        loss = F.cosine_similarity(z_adv, z_orig, dim=1).mean()

        identity_model.zero_grad(set_to_none=True)
        if adv.grad is not None:
            adv.grad.zero_()
        loss.backward()

        with torch.no_grad():
            adv = adv - alpha * mask * adv.grad.sign()
            adv = project_linf(adv, x, eps).detach()

    return adv


# ============================================================
# 5. Run ablation study
# ============================================================

print("\n" + "="*80)
print("[*] Running FaceGuard ablation study")
print("="*80)

ablation_rows = []

for seed in SEEDS:
    set_seed(seed)
    print(f"\n[*] Ablation seed {seed}")

    for idx, pair in enumerate(tqdm(pairs, desc=f"Seed {seed}")):
        src_rgb = pair["source_rgb"]
        x = rgb_to_tensor(src_rgb)
        M = make_face_mask(IMAGE_SIZE, IMAGE_SIZE, mode="ellipse")

        for eps in EPSILON_LIST:
            for alpha_ratio in ALPHA_RATIOS:

                methods = {
                    "Random-Noise-Sanity": ablation_random_noise(x, eps, mask=M),
                    "Ablation-FGSM-1Step": ablation_fgsm(x, eps, alpha_ratio=alpha_ratio, mask=M),
                    "Ablation-PGD-NoEOT": ablation_pgd_no_eot(x, eps, alpha_ratio=alpha_ratio, steps=STEPS, mask=M),
                    "FaceGuard-EOT": protect_faceguard(
                        x, eps, alpha_ratio=alpha_ratio, steps=STEPS, mask=M,
                        use_eot=USE_EOT_FOR_FACEGUARD
                    )
                }

                for method_name, adv in methods.items():
                    cos_val = cosine_identity(x, adv)
                    psnr_val = psnr_tensor(x, adv)
                    ssim_val = ssim_np_rgb(src_rgb, tensor_to_rgb_uint8(adv))

                    adv_jpeg = simulate_jpeg_tensor(adv, quality=75)
                    adv_blur = simulate_blur_tensor(adv, ksize=5)
                    adv_resize = simulate_resize_tensor(adv, scale=0.80)

                    jpeg_cos = cosine_identity(x, adv_jpeg)
                    blur_cos = cosine_identity(x, adv_blur)
                    resize_cos = cosine_identity(x, adv_resize)

                    ablation_rows.append({
                        "seed": seed,
                        "pair_idx": idx,
                        "method": method_name,
                        "epsilon": eps,
                        "epsilon_pixel": eps * 255,
                        "alpha_ratio": alpha_ratio,
                        "identity_cosine": cos_val,
                        "psnr": psnr_val,
                        "ssim": ssim_val,
                        "jpeg_cosine": jpeg_cos,
                        "blur_cosine": blur_cos,
                        "resize_cosine": resize_cos,
                        "source_label": pair["source_label"],
                        "target_label": pair["target_label"]
                    })

ablation_df = pd.DataFrame(ablation_rows)
ablation_csv = os.path.join(OUTPUT_DIR, "faceguard_ablation_study.csv")
ablation_df.to_csv(ablation_csv, index=False)
print("[*] Saved:", ablation_csv)
display(ablation_df.head())


# ============================================================
# 6. Statistical summaries and plots
# ============================================================

def summarize_group(df, group_cols, metric_cols):
    rows = []
    for keys, sub in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)
        base = {col: val for col, val in zip(group_cols, keys)}
        for metric in metric_cols:
            mean = sub[metric].mean()
            std = sub[metric].std()
            n = len(sub)
            ci95 = 1.96 * std / np.sqrt(n) if n > 1 else 0.0
            row = base.copy()
            row.update({
                "metric": metric,
                "mean": mean,
                "std": std,
                "ci95": ci95,
                "n": n,
                "latex_mean_std": f"{mean:.3f} $\\pm$ {std:.3f}",
                "latex_mean_ci95": f"{mean:.3f} $\\pm$ {ci95:.3f}",
            })
            rows.append(row)
    return pd.DataFrame(rows)


metric_cols = ["identity_cosine", "psnr", "ssim", "jpeg_cosine", "blur_cosine", "resize_cosine"]
summary_ablation = summarize_group(
    ablation_df,
    group_cols=["method", "epsilon_pixel"],
    metric_cols=metric_cols
)

summary_csv = os.path.join(OUTPUT_DIR, "faceguard_ablation_summary_latex_ready.csv")
summary_ablation.to_csv(summary_csv, index=False)
print("[*] Saved:", summary_csv)
display(summary_ablation.head(20))


def plot_metric(df, metric, ylabel, title, save_name):
    plt.figure(figsize=(9, 6))

    methods = list(df["method"].unique())
    for method in methods:
        sub = df[df["method"] == method]
        g = sub.groupby("epsilon_pixel")[metric].agg(["mean", "std", "count"]).reset_index()
        ci95 = 1.96 * g["std"] / np.sqrt(g["count"])

        plt.errorbar(
            g["epsilon_pixel"],
            g["mean"],
            yerr=ci95,
            marker="o",
            capsize=4,
            linewidth=2,
            label=method
        )

    plt.xlabel("Perturbation Budget ε × 255")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    save_path = os.path.join(OUTPUT_DIR, save_name)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    print("[*] Saved figure:", save_path)


plot_metric(
    ablation_df,
    metric="identity_cosine",
    ylabel="Identity Cosine Similarity",
    title="Ablation Study: Identity Disruption",
    save_name="fig_ablation_identity_cosine.png"
)

plot_metric(
    ablation_df,
    metric="psnr",
    ylabel="PSNR (dB)",
    title="Ablation Study: Perceptual Fidelity",
    save_name="fig_ablation_psnr.png"
)

plot_metric(
    ablation_df,
    metric="ssim",
    ylabel="SSIM",
    title="Ablation Study: Structural Similarity",
    save_name="fig_ablation_ssim.png"
)

fg_df = ablation_df[ablation_df["method"] == "FaceGuard-EOT"].copy()

plot_metric(
    fg_df,
    metric="jpeg_cosine",
    ylabel="Identity Cosine After JPEG",
    title="FaceGuard Robustness Against JPEG Compression",
    save_name="fig_faceguard_jpeg_robustness.png"
)

plot_metric(
    fg_df,
    metric="blur_cosine",
    ylabel="Identity Cosine After Gaussian Blur",
    title="FaceGuard Robustness Against Gaussian Blur",
    save_name="fig_faceguard_blur_robustness.png"
)

plot_metric(
    fg_df,
    metric="resize_cosine",
    ylabel="Identity Cosine After Resize/Downscale",
    title="FaceGuard Robustness Against Resize/Downscale",
    save_name="fig_faceguard_resize_robustness.png"
)


# ============================================================
# 7. Save visual examples of original / protected / perturbation
# ============================================================

print("\n[*] Saving FaceGuard visual examples...")

EVAL_EPS = 12/255
EVAL_ALPHA_RATIO = 0.125

for idx, pair in enumerate(pairs[:SAVE_VISUAL_EXAMPLES]):
    src_rgb = pair["source_rgb"]
    tgt_rgb = pair["target_rgb"]
    x = rgb_to_tensor(src_rgb)
    M = make_face_mask(IMAGE_SIZE, IMAGE_SIZE, mode="ellipse")

    x_protected = protect_faceguard(
        x,
        eps=EVAL_EPS,
        alpha_ratio=EVAL_ALPHA_RATIO,
        steps=STEPS,
        mask=M,
        use_eot=USE_EOT_FOR_FACEGUARD
    )

    protected_rgb = tensor_to_rgb_uint8(x_protected)

    diff = np.abs(protected_rgb.astype(np.float32) - src_rgb.astype(np.float32))
    diff = diff / (diff.max() + 1e-8)
    diff_vis = (diff * 255).astype(np.uint8)

    cos_val = cosine_identity(x, x_protected)
    psnr_val = psnr_np_rgb(src_rgb, protected_rgb)
    ssim_val = ssim_np_rgb(src_rgb, protected_rgb)

    save_path = os.path.join(VIS_DIR, f"faceguard_protection_example_{idx:03d}.png")
    save_image_grid(
        images=[src_rgb, protected_rgb, diff_vis, tgt_rgb],
        titles=["Original Source", "Protected Source", "Perturbation Map", "Target"],
        save_path=save_path,
        suptitle=f"FaceGuard example {idx}: cosine={cos_val:.3f}, PSNR={psnr_val:.2f}, SSIM={ssim_val:.3f}",
        figsize=(16, 4)
    )


# ============================================================
# 8. Optional real deepfake evaluation with InsightFace InSwapper
# ============================================================

print("\n" + "="*80)
print("[*] Optional real deepfake evaluation")
print("="*80)

real_deepfake_ready = False
deepfake_df = pd.DataFrame()

if RUN_REAL_DEEPFAKE:
    if not os.path.exists(SWAPPER_MODEL_PATH):
        print("[!] Real deepfake evaluation is enabled, but the model file was not found:")
        print("    ", SWAPPER_MODEL_PATH)
        print("[!] Upload a valid inswapper_128.onnx to /content/ or set SWAPPER_MODEL_PATH.")
        print("[!] Skipping real deepfake evaluation. Ablation and robustness results are complete.")
    else:
        try:
            import insightface
            from insightface.app import FaceAnalysis
            from insightface import model_zoo

            providers = ["CUDAExecutionProvider", "CPUExecutionProvider"] if torch.cuda.is_available() else ["CPUExecutionProvider"]

            print("[*] Loading InsightFace FaceAnalysis...")
            face_app = FaceAnalysis(name="buffalo_l", providers=providers)
            face_app.prepare(ctx_id=0 if torch.cuda.is_available() else -1, det_size=(640, 640))

            print("[*] Loading InSwapper model...")
            swapper = model_zoo.get_model(SWAPPER_MODEL_PATH, providers=providers)
            real_deepfake_ready = True
            print("[*] Real deepfake model loaded successfully.")

        except Exception as e:
            print("[!] Failed to load real deepfake model. Skipping.")
            print("    Reason:", repr(e))
            real_deepfake_ready = False


def get_largest_face(img_bgr):
    faces = face_app.get(img_bgr)
    if faces is None or len(faces) == 0:
        return None
    faces = sorted(
        faces,
        key=lambda f: (f.bbox[2]-f.bbox[0]) * (f.bbox[3]-f.bbox[1]),
        reverse=True
    )
    return faces[0]


def run_real_face_swap(source_rgb, target_rgb):
    source_bgr = rgb_to_bgr(source_rgb)
    target_bgr = rgb_to_bgr(target_rgb)

    source_face = get_largest_face(source_bgr)
    target_face = get_largest_face(target_bgr)

    if source_face is None or target_face is None:
        return None

    result_bgr = swapper.get(target_bgr.copy(), target_face, source_face, paste_back=True)
    result_rgb = bgr_to_rgb(result_bgr)
    result_rgb = resize_rgb(result_rgb, IMAGE_SIZE)
    return result_rgb


if real_deepfake_ready:
    print("[*] Running real deepfake evaluation...")

    deepfake_rows = []
    eval_pairs = pairs[:MAX_DEEPFAKE_EVAL_PAIRS]

    for idx, pair in enumerate(tqdm(eval_pairs, desc="Real deepfake eval")):
        src_rgb = pair["source_rgb"]
        tgt_rgb = pair["target_rgb"]

        x_src = rgb_to_tensor(src_rgb)
        M = make_face_mask(IMAGE_SIZE, IMAGE_SIZE, mode="ellipse")

        x_protected = protect_faceguard(
            x_src,
            eps=EVAL_EPS,
            alpha_ratio=EVAL_ALPHA_RATIO,
            steps=STEPS,
            mask=M,
            use_eot=USE_EOT_FOR_FACEGUARD
        )
        protected_rgb = tensor_to_rgb_uint8(x_protected)

        swap_clean_rgb = run_real_face_swap(src_rgb, tgt_rgb)
        swap_protected_rgb = run_real_face_swap(protected_rgb, tgt_rgb)

        if swap_clean_rgb is None or swap_protected_rgb is None:
            deepfake_rows.append({
                "pair_idx": idx,
                "status": "face_detection_failed",
                "epsilon": EVAL_EPS,
                "epsilon_pixel": EVAL_EPS * 255,
                "alpha_ratio": EVAL_ALPHA_RATIO,
                "source_label": pair["source_label"],
                "target_label": pair["target_label"]
            })
            continue

        x_tgt = rgb_to_tensor(tgt_rgb)
        x_swap_clean = rgb_to_tensor(swap_clean_rgb)
        x_swap_protected = rgb_to_tensor(swap_protected_rgb)

        source_to_clean_swap_cos = cosine_identity(x_src, x_swap_clean)
        source_to_protected_swap_cos = cosine_identity(x_src, x_swap_protected)
        target_to_clean_swap_cos = cosine_identity(x_tgt, x_swap_clean)
        target_to_protected_swap_cos = cosine_identity(x_tgt, x_swap_protected)

        protection_drop = source_to_clean_swap_cos - source_to_protected_swap_cos

        protected_psnr = psnr_np_rgb(src_rgb, protected_rgb)
        protected_ssim = ssim_np_rgb(src_rgb, protected_rgb)

        swap_output_psnr = psnr_np_rgb(swap_clean_rgb, swap_protected_rgb)
        swap_output_ssim = ssim_np_rgb(swap_clean_rgb, swap_protected_rgb)

        deepfake_rows.append({
            "pair_idx": idx,
            "status": "ok",
            "epsilon": EVAL_EPS,
            "epsilon_pixel": EVAL_EPS * 255,
            "alpha_ratio": EVAL_ALPHA_RATIO,

            "source_to_clean_swap_cos": source_to_clean_swap_cos,
            "source_to_protected_swap_cos": source_to_protected_swap_cos,
            "protection_drop": protection_drop,

            "target_to_clean_swap_cos": target_to_clean_swap_cos,
            "target_to_protected_swap_cos": target_to_protected_swap_cos,

            "protected_source_psnr": protected_psnr,
            "protected_source_ssim": protected_ssim,

            "swap_clean_vs_protected_psnr": swap_output_psnr,
            "swap_clean_vs_protected_ssim": swap_output_ssim,

            "source_label": pair["source_label"],
            "target_label": pair["target_label"]
        })

        if idx < SAVE_VISUAL_EXAMPLES:
            save_path = os.path.join(VIS_DIR, f"real_deepfake_example_{idx:03d}.png")
            save_image_grid(
                images=[src_rgb, protected_rgb, tgt_rgb, swap_clean_rgb, swap_protected_rgb],
                titles=["Original Source", "Protected Source", "Target", "Deepfake\nNo Protection", "Deepfake\nAfter FaceGuard"],
                save_path=save_path,
                suptitle=(
                    f"Pair {idx}: source-to-swap cosine clean={source_to_clean_swap_cos:.3f}, "
                    f"protected={source_to_protected_swap_cos:.3f}, drop={protection_drop:.3f}"
                ),
                figsize=(18, 4)
            )

    deepfake_df = pd.DataFrame(deepfake_rows)
    deepfake_csv = os.path.join(OUTPUT_DIR, "faceguard_real_deepfake_eval.csv")
    deepfake_df.to_csv(deepfake_csv, index=False)
    print("[*] Saved:", deepfake_csv)
    display(deepfake_df.head())

    ok_df = deepfake_df[deepfake_df["status"] == "ok"].copy()

    if len(ok_df) > 0:
        real_metric_cols = [
            "source_to_clean_swap_cos",
            "source_to_protected_swap_cos",
            "protection_drop",
            "target_to_clean_swap_cos",
            "target_to_protected_swap_cos",
            "protected_source_psnr",
            "protected_source_ssim",
            "swap_clean_vs_protected_psnr",
            "swap_clean_vs_protected_ssim"
        ]

        real_summary_df = summarize_group(
            ok_df,
            group_cols=["epsilon_pixel"],
            metric_cols=real_metric_cols
        )

        real_summary_csv = os.path.join(OUTPUT_DIR, "faceguard_real_deepfake_summary_latex_ready.csv")
        real_summary_df.to_csv(real_summary_csv, index=False)

        print("[*] Successful real deepfake samples:", len(ok_df))
        print("[*] Saved:", real_summary_csv)
        display(real_summary_df)

        plt.figure(figsize=(7, 5))
        vals = [
            ok_df["source_to_clean_swap_cos"].mean(),
            ok_df["source_to_protected_swap_cos"].mean()
        ]
        errs = [
            1.96 * ok_df["source_to_clean_swap_cos"].std() / np.sqrt(len(ok_df)),
            1.96 * ok_df["source_to_protected_swap_cos"].std() / np.sqrt(len(ok_df))
        ]

        plt.bar(["No Protection", "FaceGuard Protected"], vals, yerr=errs, capsize=5)
        plt.ylabel("Source Identity Cosine in Deepfake Output")
        plt.title("Real Deepfake Evaluation: Identity Transfer Suppression")
        plt.grid(axis="y", linestyle="--", alpha=0.5)

        fig_path = os.path.join(OUTPUT_DIR, "fig_real_deepfake_identity_transfer.png")
        plt.savefig(fig_path, dpi=300, bbox_inches="tight")
        plt.show()
        plt.close()

        print("[*] Saved figure:", fig_path)
    else:
        print("[!] No successful real deepfake samples. Check face detection/model compatibility.")


# ============================================================
# 9. Final report and zip outputs
# ============================================================

print("\n" + "="*80)
print("[*] Final files generated")
print("="*80)

for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = "  " * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

zip_path = "/content/faceguard_outputs.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive("/content/faceguard_outputs", "zip", OUTPUT_DIR)
print("\n[*] Zipped outputs:", zip_path)

print("\n" + "="*80)
print("[*] DONE")
print("="*80)
print("Main outputs:")
print("1) /content/faceguard_outputs/faceguard_ablation_study.csv")
print("2) /content/faceguard_outputs/faceguard_ablation_summary_latex_ready.csv")
print("3) /content/faceguard_outputs/visual_examples/")
print("4) /content/faceguard_outputs.zip")
if real_deepfake_ready:
    print("5) /content/faceguard_outputs/faceguard_real_deepfake_eval.csv")
    print("6) /content/faceguard_outputs/faceguard_real_deepfake_summary_latex_ready.csv")
else:
    print("Real deepfake evaluation was skipped because inswapper_128.onnx was not available or failed to load.")

[*] Device: cuda
[*] Output directory: /content/faceguard_outputs
[*] Real deepfake model path: /content/inswapper_128.onnx
[*] Loading LFW dataset...
[*] Loaded 80 LFW source-target pairs from sklearn.
[*] Example source-target shapes: (224, 224, 3) (224, 224, 3)
[*] Loading identity extractor: InceptionResnetV1 pretrained on VGGFace2...


  0%|          | 0.00/107M [00:00<?, ?B/s]

[*] Identity extractor loaded.

[*] Running FaceGuard ablation study

[*] Ablation seed 0


Seed 0:   2%|▎         | 2/80 [06:06<3:58:17, 183.30s/it]


KeyboardInterrupt: 